In [ ]:
!pip install scikit-surprise


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 6.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# Install and import necessary packages
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
import kagglehub
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split, GridSearchCV


# Download the latest version of the MovieLens 20M dataset
path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")
print("Path to dataset files:", path)

# Load the ratings and movies data
ratings_df = pd.read_csv(f"{path}/rating.csv")
movies_df = pd.read_csv(f"{path}/movie.csv")

# Keep users with at least 20 ratings and sample 5000 users
min_ratings = 20
user_counts = ratings_df['userId'].value_counts()
active_users = user_counts[user_counts >= min_ratings].index
ratings_df = ratings_df[ratings_df['userId'].isin(active_users)]
sample_users = ratings_df['userId'].unique()[:5000]
ratings_df = ratings_df[ratings_df['userId'].isin(sample_users)]

print(f"Filtered dataset: {len(ratings_df)} ratings")


def train_test_split_by_user(ratings, test_size=0.1):
    train_list, test_list = [], []
    for _, group in ratings.groupby('userId'):
        n_test = max(1, int(test_size * len(group)))
        test_indices = group.sample(n=n_test, random_state=42).index
        train_list.append(group.drop(test_indices))
        test_list.append(group.loc[test_indices])
    return pd.concat(train_list), pd.concat(test_list)

train_df, test_df = train_test_split_by_user(ratings_df, test_size=0.1)
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")


unique_users = train_df['userId'].unique()
unique_movies = train_df['movieId'].unique()
user_to_index = {user: idx for idx, user in enumerate(unique_users)}
movie_to_index = {movie: idx for idx, movie in enumerate(unique_movies)}
index_to_movie = {idx: movie for movie, idx in movie_to_index.items()}

num_users, num_movies = len(unique_users), len(unique_movies)

row_indices = train_df['userId'].map(user_to_index)
col_indices = train_df['movieId'].map(movie_to_index)
data = train_df['rating']
train_matrix = csr_matrix((data, (row_indices, col_indices)), shape=(num_users, num_movies))

latent_factors = 100  # Increased for better representation
user_means = train_matrix.mean(axis=1).A1  # Convert sparse to array
train_matrix_centered = train_matrix - user_means.reshape(-1, 1)

U, sigma, Vt = svds(train_matrix_centered, k=latent_factors)
predicted_ratings = np.dot(np.dot(U, np.diag(sigma)), Vt) + user_means.reshape(-1, 1)

def get_svd_recommendations(user_id, predicted_ratings, k=10):
    if user_id not in user_to_index:
        return []
    user_idx = user_to_index[user_id]
    user_predicted = predicted_ratings[user_idx].copy()

    # Exclude already rated items
    user_rated_items = train_matrix[user_idx].nonzero()[1]
    user_predicted[user_rated_items] = -np.inf

    recommended_indices = np.argsort(user_predicted)[::-1][:k]
    return [index_to_movie[idx] for idx in recommended_indices]


def precision_recall_at_k(recommended, relevant):
    recommended_set, relevant_set = set(recommended), set(relevant)
    hits = len(recommended_set & relevant_set)
    precision = hits / len(recommended) if recommended else 0
    recall = hits / len(relevant) if relevant else 0
    return precision, recall

relevant_threshold = 4.0
test_user_relevant = {user: group[group['rating'] >= relevant_threshold]['movieId'].tolist()
                      for user, group in test_df.groupby('userId')}


k = 10
svd_precisions, svd_recalls = [], []

evaluated_users = [user for user in test_user_relevant if user in user_to_index]
print(f"Evaluating {len(evaluated_users)} users...")

for user in evaluated_users:
    relevant = test_user_relevant[user]
    svd_recs = get_svd_recommendations(user, predicted_ratings, k=k)
    p_svd, r_svd = precision_recall_at_k(svd_recs, relevant)
    svd_precisions.append(p_svd)
    svd_recalls.append(r_svd)

avg_precision_svd = np.mean(svd_precisions)
avg_recall_svd = np.mean(svd_recalls)

print("\nImproved SVD-Based Collaborative Filtering:")
print(f"Precision@{k}: {avg_precision_svd:.4f}")
print(f"Recall@{k}: {avg_recall_svd:.4f}")

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings_df[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2)

param_grid = {'n_factors': [50, 100, 150], 'n_epochs': [20, 30, 50], 'lr_all': [0.002, 0.005, 0.01], 'reg_all': [0.02, 0.05, 0.1]}
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs.fit(data)

best_params = gs.best_params['rmse']
print("\nBest hyperparameters for SVD:", best_params)

# Train optimized SVD
best_svd = SVD(n_factors=best_params['n_factors'], n_epochs=best_params['n_epochs'],
               lr_all=best_params['lr_all'], reg_all=best_params['reg_all'])
best_svd.fit(trainset)

# Evaluate optimized SVD
predictions = best_svd.test(testset)
rmse = np.sqrt(np.mean([(pred.est - true) ** 2 for (_, _, true, pred.est, _) in predictions]))
print(f"Optimized SVD RMSE: {rmse:.4f}")
